# Pose Estimator Test
This file is a test script of MediaPipe's Pose Estimator on pre-recorded video.

In [1]:
import numpy as np
import pandas as pd
import cv2
import os
import math


import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from mediapipe.tasks.python.vision import drawing_utils
from mediapipe.tasks.python.vision import drawing_styles

In [2]:
def draw_landmarks_on_image(rgb_image, detection_result):
  pose_landmarks_list = detection_result.pose_landmarks
  annotated_image = np.copy(rgb_image)

  pose_landmark_style = drawing_styles.get_default_pose_landmarks_style()
  pose_connection_style = drawing_utils.DrawingSpec(color=(0, 255, 0), thickness=2)

  for pose_landmarks in pose_landmarks_list:
    drawing_utils.draw_landmarks(
        image=annotated_image,
        landmark_list=pose_landmarks,
        connections=vision.PoseLandmarksConnections.POSE_LANDMARKS,
        landmark_drawing_spec=pose_landmark_style,
        connection_drawing_spec=pose_connection_style)

  return annotated_image

if not os.path.exists('pose_landmarker.task'):
    import urllib.request
    url = 'https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task'
    urllib.request.urlretrieve(url, 'pose_landmarker.task')

# STEP 2: Create an PoseLandmarker object.
base_options = python.BaseOptions(model_asset_path='pose_landmarker.task')
options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    output_segmentation_masks=True)
detector = vision.PoseLandmarker.create_from_options(options)



In [3]:
def calculate_angle(a, b, c):
    # a, b, c are NormalizedLandmark objects with x, y, z
    ba = np.array([a.x - b.x, a.y - b.y, a.z - b.z])
    bc = np.array([c.x - b.x, c.y - b.y, c.z - b.z])
    cosine_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc))
    angle = np.arccos(np.clip(cosine_angle, -1, 1))
    return np.degrees(angle)

def estimate_joint_angles(detection_result):
    if not detection_result.pose_landmarks:
        return {}
    
    landmarks = detection_result.pose_landmarks[0]  # assume one person
    
    angles = {}
    
    # Left elbow: shoulder (11), elbow (13), wrist (15)
    angles['left_elbow'] = calculate_angle(landmarks[11], landmarks[13], landmarks[15])
    
    # Right elbow: shoulder (12), elbow (14), wrist (16)
    angles['right_elbow'] = calculate_angle(landmarks[12], landmarks[14], landmarks[16])
    
    # Left knee: hip (23), knee (25), ankle (27)
    angles['left_knee'] = calculate_angle(landmarks[23], landmarks[25], landmarks[27])
    
    # Right knee: hip (24), knee (26), ankle (28)
    angles['right_knee'] = calculate_angle(landmarks[24], landmarks[26], landmarks[28])
    
    return angles

def display_angles_on_image(image, angles):
    """Display angle values as text on the image"""
    y_offset = 30
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.7
    color = (0, 255, 0)  # Green text
    thickness = 2
    
    for joint_name, angle_value in angles.items():
        text = f"{joint_name}: {angle_value:.2f}°"
        cv2.putText(image, text, (10, y_offset), font, font_scale, color, thickness)
        y_offset += 30
    
    return image

In [4]:
# Live feed

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise RuntimeError("Webcam could not be opened.")

while cap.isOpened():
  ret, frame = cap.read()
  if not ret:
    print("Frame grab failed.")
    break

  # BGR to RGB
  rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
  image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

  detection_result = detector.detect(image)

  # Estimate joint angles
  angles = estimate_joint_angles(detection_result)

  landmark_image = draw_landmarks_on_image(image.numpy_view(), detection_result)
  
  # Display angles on the image
  landmark_image = display_angles_on_image(landmark_image, angles)
  
  output = cv2.cvtColor(landmark_image, cv2.COLOR_RGB2BGR)

  cv2.imshow("live pose estimation", output)

  # q to quit
  if cv2.waitKey(1) & 0xFF == ord('q'):
    break

cap.release()
cv2.destroyAllWindows()